# 03 · Substituição do modelo subjacente

O laço de controle independe do modelo: a lógica do agente não se altera ao trocar o backend.
Este notebook examina as consequências práticas dessa substituição e, em particular, uma
limitação frequentemente subestimada.

Nem todo modelo implementa *function calling* estruturado, e a forma como essa limitação se
manifesta depende da plataforma:

- Em modelos servidos **localmente** (p. ex., via Ollama), um modelo sem esse suporte tende a
  descrever a ação pretendida em linguagem natural (p. ex., "utilizarei a calculadora...").
  `message.tool_calls` permanece vazio e o agente retorna o texto sem executar ferramenta alguma.
- Em endpoints **hospedados**, como o da NVIDIA empregado aqui, a mesma incapacidade costuma
  manifestar-se como **rejeição da requisição** (erro HTTP 4xx): o parâmetro `tool_choice`, ou a
  própria capacidade de *tool calling*, não é aceito pelo *deployment* do modelo.

Em ambos os casos vale a mesma heurística diagnóstica: se o agente nunca executa ferramentas
— por retornar texto ou por levantar exceção —, a inadequação está, em primeira instância, no
modelo, não no código. O `run_agent` abaixo trata exceções para que o experimento prossiga e o
erro seja, ele próprio, um dado.


## Dependências

In [ ]:
!pip install -q openai

## Configuração do cliente

Empregamos a biblioteca `openai` apontando para o endpoint **gratuito da NVIDIA**
(`https://integrate.api.nvidia.com/v1`), que expõe uma interface compatível com a API da OpenAI.
A célula seguinte concentra os parâmetros de acesso: endpoint, modelo e credencial.

Para obter a credencial: criar conta em https://build.nvidia.com e gerar uma API key (prefixo
`nvapi-`). No Google Colab, a chave pode ser guardada uma única vez em *Secrets* (ícone de chave na
barra lateral) com o nome `NVIDIA_API_KEY`; a célula a seguir a detecta automaticamente. Para
utilizar outro modelo, basta substituir a variável `MODEL` por outro identificador do catálogo
(https://build.nvidia.com/models) — nenhum outro ponto do código precisa mudar.


In [ ]:
import os
import json
from getpass import getpass
from datetime import datetime
from openai import OpenAI

def _obter_api_key():
    try:                                  # 1) Colab Secrets (icone de chave a esquerda)
        from google.colab import userdata
        chave = userdata.get("NVIDIA_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    return os.environ.get("NVIDIA_API_KEY") or getpass("API key (nvapi-...): ")  # 2) env var ou 3) manual

# Parametros de backend (alterar para usar outro provedor)
BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL    = "meta/llama-3.1-8b-instruct"
API_KEY  = _obter_api_key()

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Cliente configurado:", BASE_URL, "| modelo:", MODEL)


## Ferramentas

Reutilizamos um subconjunto das ferramentas do notebook anterior.

In [ ]:
def get_current_date() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

TOOL_FUNCTIONS = {"get_current_date": get_current_date, "calculate": calculate}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_current_date",
        "description": "Retorna a data e hora atuais.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Avalia uma expressao matematica.",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string", "description": "ex.: '847 * 0.15'"}},
            "required": ["expression"]}}},
]


## O laço parametrizado pelo modelo

`run_agent` passa a receber o identificador do modelo como argumento, permitindo avaliar diversos
modelos sem alterar o restante do código. Contabilizamos também o número de invocações de
ferramenta, métrica que serve para detectar o caso em que o modelo não emite nenhuma.


In [ ]:
def run_agent(task: str, model: str, verbose: bool = False):
    messages = [
        {"role": "system", "content": "Voce e um assistente util. Use ferramentas quando precisar. Quando tiver a resposta final, responda sem chamar ferramentas."},
        {"role": "user", "content": task},
    ]
    n_tool_calls = 0
    while True:
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, tool_choice="auto",
        )
        message = response.choices[0].message
        if message.tool_calls and len(message.tool_calls) > 1:
            message.tool_calls = message.tool_calls[:1]  # este endpoint aceita 1 tool call por turno
        messages.append(message)
        if not message.tool_calls:
            return message.content, n_tool_calls
        for tool_call in message.tool_calls:
            n_tool_calls += 1
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"   -> {name}({args})")
            fn = TOOL_FUNCTIONS.get(name)
            result = fn(**args) if fn else f"Ferramenta desconhecida: {name}"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})


## Comparação entre modelos

Submetemos uma mesma tarefa — cuja resolução demanda cálculo — a diferentes modelos do catálogo
da NVIDIA, registrando o número de ferramentas efetivamente invocadas por cada um. Os dois
primeiros suportam *function calling* e invocam a calculadora; o último (`gemma-2-2b-it`) não o
suporta e, neste endpoint hospedado, **rejeita a requisição** — capturamos a exceção e a
exibimos, pois ela é a evidência da limitação.

Os identificadores e as latências foram verificados no catálogo (https://build.nvidia.com/models)
em 2026-06; a disponibilidade e o tempo de resposta podem variar conforme a conta e a carga do
endpoint (incluindo eventuais *cold starts*).


In [ ]:
modelos = [
    "meta/llama-3.1-8b-instruct",         # suporta function calling (~1s)
    "nvidia/nvidia-nemotron-nano-9b-v2",  # suporta (um pouco mais lento)
    "google/gemma-2-2b-it",               # NAO suporta -> rejeita a requisicao
]

tarefa = "Quanto e 15% de 847? Utilize a calculadora."

for m in modelos:
    try:
        resposta, n = run_agent(tarefa, model=m)
        status = "invocou ferramenta" if n > 0 else "NENHUMA invocacao (texto puro)"
        print(f"[{n} invocacoes] {status}\n   modelo  : {m}\n   resposta: {resposta}\n")
    except Exception as e:
        print(f"[rejeitado] {m}\n   {type(e).__name__}: {str(e)[:120]}\n")


## Substituição do modelo: um único ponto de mudança

A independência do agente em relação ao modelo torna a substituição trivial: altera-se apenas a
variável `MODEL` na célula de configuração, e o `run_agent` permanece intacto. Dentro do catálogo
da NVIDIA, isso significa alternar livremente entre os modelos verificados acima como compatíveis
com *function calling* — por exemplo:

```python
MODEL = "meta/llama-3.1-8b-instruct"          # default da disciplina (~1s por chamada)
MODEL = "nvidia/nvidia-nemotron-nano-9b-v2"   # alternativa, um pouco mais lenta
MODEL = "meta/llama-3.1-70b-instruct"         # modelo maior: mais latencia, sujeito a cold start
```

Reexecute a célula de configuração após a troca e rode novamente as tarefas dos notebooks
anteriores: o comportamento do agente se altera, mas não uma única linha de `run_agent`. É essa
separação entre o laço de controle e o modelo que torna o agente um artefato reutilizável.


---
## Síntese da sequência

- **01** — o agente como laço de controle: um `while` que alterna entre modelo e ferramentas.
- **02** — composição de ferramentas e o histórico como memória de trabalho.
- **03** — o modelo como componente substituível, sujeito à condição de suportar *function calling*.

Os mecanismos ausentes nesta implementação — tratamento de falhas, *retry*, limites de iteração,
delegação entre modelos, interoperabilidade via MCP — são precisamente o que justifica frameworks
como LangGraph, CrewAI ou o Claude Agents SDK. A construção da versão elementar fornece, contudo,
o modelo mental necessário para avaliar criticamente o que essas camadas adicionam.
